# 🤖 Notebook 2: Entrenamiento del Modelo

Este notebook cubre:
- Carga de datos con DataLoaders
- Creación del modelo con Transfer Learning (MobileNetV2)
- Configuración del entrenamiento
- Entrenamiento con early stopping
- Guardado del modelo

In [ ]:
# Imports
import sys
sys.path.append('..')

import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from src.dataset.dataset import create_dataloaders
from src.models.classifier import create_model
from src.models.train import train_model, EarlyStopping
from src.utils.helpers import load_config, plot_training_history

# Configuración
plt.rcParams['figure.figsize'] = (15, 5)

## 1. Configuración

In [ ]:
# Cargar configuración
config = load_config('../config/config.yaml')

# Parámetros
BATCH_SIZE = config['training']['batch_size']
EPOCHS = config['training']['epochs']
LEARNING_RATE = config['training']['learning_rate']
IMAGE_SIZE = config['dataset']['image_size']
ARCHITECTURE = config['model']['architecture']

# Dispositivo
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")
print(f"PyTorch version: {torch.__version__}")

## 2. Carga de Datos

In [ ]:
# Crear DataLoaders
train_loader, val_loader, test_loader, class_names = create_dataloaders(
    train_dir='../data/processed/train',
    val_dir='../data/processed/val',
    test_dir='../data/processed/test',
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    num_workers=2,  # Ajustar según tu CPU
    augment=True
)

num_classes = len(class_names)
print(f"\nClases ({num_classes}):")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

## 3. Visualización de Datos Aumentados

In [ ]:
# Visualizar batch de entrenamiento
import torchvision
import numpy as np

def imshow(img, title=None):
    """Muestra una imagen de tensor."""
    # Desnormalizar
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose((1, 2, 0))
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')

# Obtener un batch
images, labels = next(iter(train_loader))

# Mostrar grid
fig = plt.figure(figsize=(15, 8))
for i in range(min(8, len(images))):
    ax = plt.subplot(2, 4, i + 1)
    imshow(images[i], title=class_names[labels[i]])

plt.tight_layout()
plt.show()

## 4. Creación del Modelo

In [ ]:
# Crear modelo
model = create_model(
    num_classes=num_classes,
    architecture=ARCHITECTURE,
    pretrained=True,
    dropout=config['model']['dropout'],
    freeze_ratio=config['model']['freeze_ratio'],
    device=device
)

## 5. Configuración del Entrenamiento

In [ ]:
# Función de pérdida
criterion = nn.CrossEntropyLoss()

# Optimizador
if config['training']['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
else:
    optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)

# Learning rate scheduler
if config['training']['scheduler'] == 'step':
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=config['training']['step_size'],
        gamma=config['training']['gamma']
    )
else:
    scheduler = None

# Early stopping
early_stopping = None
if config['training']['early_stopping']['enabled']:
    early_stopping = EarlyStopping(
        patience=config['training']['early_stopping']['patience'],
        mode='min'
    )

print("Configuración de entrenamiento:")
print(f"  - Función de pérdida: CrossEntropyLoss")
print(f"  - Optimizador: {config['training']['optimizer']}")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Scheduler: {config['training']['scheduler']}")
print(f"  - Early stopping: {config['training']['early_stopping']['enabled']}")

## 6. Entrenamiento

In [ ]:
# Entrenar modelo
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=EPOCHS,
    device=device,
    scheduler=scheduler,
    early_stopping=early_stopping,
    checkpoint_dir='../models/checkpoints',
    class_names=class_names
)

## 7. Visualización del Entrenamiento

In [ ]:
# Graficar historial de entrenamiento
plot_training_history(history, save_path='../docs/training_history.png')

## 8. Guardar Modelo Final

In [ ]:
# Copiar el mejor modelo a la ubicación final
import shutil

best_model_path = '../models/checkpoints/best_model.pth'
final_model_path = '../models/best_model.pth'

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, final_model_path)
    print(f"✓ Modelo guardado en: {final_model_path}")
else:
    print("⚠ No se encontró el mejor modelo")

## 9. Resumen del Entrenamiento

In [ ]:
# Mostrar resumen
print("\n" + "="*70)
print("RESUMEN DEL ENTRENAMIENTO")
print("="*70)
print(f"Arquitectura: {ARCHITECTURE}")
print(f"Número de clases: {num_classes}")
print(f"Épocas entrenadas: {len(history['train_loss'])}")
print(f"Mejor Train Acc: {max(history['train_acc']):.2f}%")
print(f"Mejor Val Acc: {max(history['val_acc']):.2f}%")
print(f"Mejor Train Loss: {min(history['train_loss']):.4f}")
print(f"Mejor Val Loss: {min(history['val_loss']):.4f}")
print("="*70)

## 10. Próximos Pasos

1. ✅ Modelo entrenado y guardado
2. ➡️ Continuar con `03_evaluacion_modelo.ipynb` para evaluar en el conjunto de test
3. ➡️ Probar el modelo en la aplicación Flask

In [ ]:
print("\n✓ Entrenamiento completado")
print("\nPróximo paso: Abrir 03_evaluacion_modelo.ipynb")